<a href="https://www.kaggle.com/code/samithsachidanandan/churn-prediction-xgb-lgbm-ensemble?scriptVersionId=301578408" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, roc_curve, auc, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import TargetEncoder

from sklearn.model_selection import ParameterSampler
from scipy.optimize import minimize
import random

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from scipy.optimize import minimize_scalar

import warnings
warnings.filterwarnings('ignore')
#sns.set_theme(style="whitegrid")

# Load the Data

In [ ]:
# Load the data

train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')
orig = pd.read_csv('/kaggle/input/datasets/cdeotte/s6e3-original-dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Standardize Target
train['Churn'] = train['Churn'].map({'No': 0, 'Yes': 1})
orig['Churn'] = orig['Churn'].map({'No': 0, 'Yes': 1})

# THE FIX: Prevent "TotalCharges" numeric errors
for df in [train, test, orig]:
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
    df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'], errors='coerce').fillna(0)
    df['tenure'] = pd.to_numeric(df['tenure'], errors='coerce').fillna(0)

CATS = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 
        'InternetService', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 
        'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
NUMS = ['tenure', 'MonthlyCharges', 'TotalCharges']


In [ ]:
train.describe().style.background_gradient(cmap='tab20c')

# Feature Engineering

In [ ]:


NEW_NUMS = []


for df in [train, test, orig]:
    df['ChargesPerMonth']       = df['TotalCharges'] / (df['tenure'] + 1)
    df['AvgMonthlyOverContract'] = df['MonthlyCharges'] / (df['tenure'] + 1)
    df['TenureGroup']           = pd.cut(df['tenure'], bins=[0,12,24,48,72], labels=False).astype('float32')
    df['HasMultipleServices']   = (
        (df['MultipleLines']    == 'Yes').astype(int) +
        (df['StreamingTV']      == 'Yes').astype(int) +
        (df['StreamingMovies']  == 'Yes').astype(int) +
        (df['OnlineBackup']     == 'Yes').astype(int) +
        (df['DeviceProtection'] == 'Yes').astype(int) +
        (df['TechSupport']      == 'Yes').astype(int)
    )

INTERACTION_FEATURES = ['ChargesPerMonth', 'AvgMonthlyOverContract', 'TenureGroup', 'HasMultipleServices']
NUMS_EXTENDED = NUMS + INTERACTION_FEATURES


for col in NUMS_EXTENDED:
    freq = pd.concat([train[col], orig[col], test[col]]).value_counts(normalize=True)
    for df in [train, test, orig]:
        df[f'FREQ_{col}'] = df[col].map(freq).fillna(0).astype('float32')
    NEW_NUMS.append(f'FREQ_{col}')


for col in CATS + NUMS_EXTENDED:
    _new_col = f"ORIG_proba_{col}"
    tmp = orig.groupby(col)['Churn'].mean().reset_index().rename(columns={'Churn': _new_col})

    train = train.merge(tmp, on=col, how="left")
    test  = test.merge(tmp, on=col, how="left")

    train[_new_col] = train[_new_col].fillna(0.5)
    test[_new_col]  = test[_new_col].fillna(0.5)
    NEW_NUMS.append(_new_col)


FEATURES = NUMS_EXTENDED + CATS + NEW_NUMS


for df in [train, test]:
    df[CATS] = df[CATS].astype(str).astype('category')

# Model Training 

In [ ]:
xgb_params = {
    'n_estimators': 50000, 
    'learning_rate': 0.01,         
    'max_depth': 5,                 
    'min_child_weight': 3,           
    'gamma': 0.1,                   
    'subsample': 0.75, 
    'colsample_bytree': 0.75,
    'colsample_bylevel': 0.75,     
    'reg_alpha': 0.1,                
    'reg_lambda': 1.5,             
    'random_state': 11,
    'early_stopping_rounds': 200,   
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'enable_categorical': True,
    'device': 'cuda',
    'tree_method': 'hist',          
}
lgb_params = {
    'n_estimators': 50000,
    'learning_rate': 0.01,
    'max_depth': 5,
    'min_child_samples': 20,
    'min_split_gain': 0.1,
    'subsample': 0.75,
    'subsample_freq': 1,
    'colsample_bytree': 0.75,
    'reg_alpha': 0.1,
    'reg_lambda': 1.5,
    'random_state': 11,
    'objective': 'binary',
    'metric': 'auc',
    'device': 'gpu',
    'verbosity': -1,
}

FOLDS      = 5
INNER_FOLDS = 5

kf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=11)

oof_xgb         = np.zeros(len(train))
oof_lgb         = np.zeros(len(train))
pred_xgb        = np.zeros(len(test))
pred_lgb        = np.zeros(len(test))
importances_xgb = []
importances_lgb = []
best_iterations_xgb = []
best_iterations_lgb = []
fold_aucs_xgb   = []
fold_aucs_lgb   = []
fold_aucs_ens   = []

for i, (train_index, val_index) in enumerate(kf.split(train, train['Churn'])):
    print(f"\n{'='*50}")
    print(f"  Fold {i+1}/{FOLDS}")
    print(f"{'='*50}")

    X_train = train.loc[train_index, FEATURES].copy()
    y_train = train.loc[train_index, 'Churn'].values
    X_val   = train.loc[val_index,   FEATURES].copy()
    y_val   = train.loc[val_index,   'Churn'].values
    X_test  = test[FEATURES].copy()

    te = TargetEncoder(cv=INNER_FOLDS, random_state=11)
    X_train[CATS] = te.fit_transform(X_train[CATS], y_train)
    X_val[CATS]   = te.transform(X_val[CATS])
    X_test[CATS]  = te.transform(X_test[CATS])


    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    best_iter_xgb = xgb_model.best_iteration
    best_iterations_xgb.append(best_iter_xgb)
    xgb_val_pred  = xgb_model.predict_proba(X_val)[:, 1]
    xgb_test_pred = xgb_model.predict_proba(X_test)[:, 1]
    importances_xgb.append(xgb_model.get_booster().get_score(importance_type='gain'))
    fold_aucs_xgb.append(roc_auc_score(y_val, xgb_val_pred))
    print(f"  XGB best iter  : {best_iter_xgb}")
    print(f"  XGB Fold AUC   : {fold_aucs_xgb[-1]:.5f}")


    lgb_model = lgb.LGBMClassifier(**lgb_params)
    lgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(period=-1)],
    )
    best_iter_lgb = lgb_model.best_iteration_
    best_iterations_lgb.append(best_iter_lgb)
    lgb_val_pred  = lgb_model.predict_proba(X_val)[:, 1]
    lgb_test_pred = lgb_model.predict_proba(X_test)[:, 1]
    importances_lgb.append(dict(zip(FEATURES, lgb_model.feature_importances_)))
    fold_aucs_lgb.append(roc_auc_score(y_val, lgb_val_pred))
    print(f"  LGB best iter  : {best_iter_lgb}")
    print(f"  LGB Fold AUC   : {fold_aucs_lgb[-1]:.5f}")


    oof_xgb[val_index] = xgb_val_pred
    oof_lgb[val_index] = lgb_val_pred
    pred_xgb          += xgb_test_pred / FOLDS
    pred_lgb          += lgb_test_pred / FOLDS

    ens_val_pred = 0.5 * xgb_val_pred + 0.5 * lgb_val_pred
    fold_aucs_ens.append(roc_auc_score(y_val, ens_val_pred))
    print(f"  Ensemble AUC   : {fold_aucs_ens[-1]:.5f}")

print(f"\n{'='*50}")
print(f"  XGB  OOF AUC   : {roc_auc_score(train['Churn'], oof_xgb):.5f}")
print(f"  LGB  OOF AUC   : {roc_auc_score(train['Churn'], oof_lgb):.5f}")
print(f"  XGB  Mean AUC  : {np.mean(fold_aucs_xgb):.5f}  ±  {np.std(fold_aucs_xgb):.5f}")
print(f"  LGB  Mean AUC  : {np.mean(fold_aucs_lgb):.5f}  ±  {np.std(fold_aucs_lgb):.5f}")
print(f"  Ens  Mean AUC  : {np.mean(fold_aucs_ens):.5f}  ±  {np.std(fold_aucs_ens):.5f}")
print(f"  XGB  mean iter : {np.mean(best_iterations_xgb):.0f}")
print(f"  LGB  mean iter : {np.mean(best_iterations_lgb):.0f}")
print(f"{'='*50}")

In [ ]:
result = minimize_scalar(
    lambda w: -roc_auc_score(train['Churn'], w * oof_xgb + (1 - w) * oof_lgb),
    bounds=(0, 1), method='bounded'
)
best_w = result.x
print(f"\n  Optimal XGB weight : {best_w:.4f}")
print(f"  Optimal LGB weight : {1 - best_w:.4f}")

oof_final  = best_w * oof_xgb  + (1 - best_w) * oof_lgb
pred_final = best_w * pred_xgb + (1 - best_w) * pred_lgb
print(f"  Optimised OOF AUC  : {roc_auc_score(train['Churn'], oof_final):.5f}")




# Visual Dashboard

In [ ]:
# Visual Dashboard

def plot_dashboard(y_true, oof_preds, test_preds, imp_xgb, imp_lgb, feat_names):
    fig, axes = plt.subplots(2, 3, figsize=(24, 12))
    fig.suptitle('Model Dashboard', fontsize=18, fontweight='bold', color='midnightblue', y=1.01)


    fpr, tpr, _ = roc_curve(y_true, oof_preds)
    axes[0, 0].plot(fpr, tpr, color='darkorange', lw=3, label=f'Ensemble AUC = {auc(fpr, tpr):.4f}')
    axes[0, 0].plot([0, 1], [0, 1], color='navy', linestyle='--')
    axes[0, 0].set_title('ROC Curve', fontsize=14, fontweight='bold', color='midnightblue')
    axes[0, 0].set_xlabel('False Positive Rate')
    axes[0, 0].set_ylabel('True Positive Rate')
    axes[0, 0].legend()


    xgb_imps = pd.DataFrame(imp_xgb).fillna(0).mean().sort_values(ascending=False).head(15)
    sns.barplot(x=xgb_imps.values, y=xgb_imps.index, ax=axes[0, 1], palette='viridis')
    axes[0, 1].set_title('XGBoost Top 15 Features (Gain)', fontsize=14, fontweight='bold', color='midnightblue')
    axes[0, 1].set_xlabel('Mean Gain')


    lgb_imps = pd.DataFrame(imp_lgb).fillna(0).mean().sort_values(ascending=False).head(15)
    sns.barplot(x=lgb_imps.values, y=lgb_imps.index, ax=axes[0, 2], palette='magma')
    axes[0, 2].set_title('LightGBM Top 15 Features (Gain)', fontsize=14, fontweight='bold', color='midnightblue')
    axes[0, 2].set_xlabel('Mean Gain')


    cm = confusion_matrix(y_true, (oof_preds > 0.5).astype(int))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0],
                xticklabels=['No Churn', 'Churn'],
                yticklabels=['No Churn', 'Churn'])
    axes[1, 0].set_title('Confusion Matrix (OOF)', fontsize=14, fontweight='bold', color='midnightblue')
    axes[1, 0].set_xlabel('Predicted')
    axes[1, 0].set_ylabel('Actual')


    sns.histplot(oof_preds[y_true == 0], bins=50, kde=True, color='steelblue',
                 label='No Churn', ax=axes[1, 1], alpha=0.6)
    sns.histplot(oof_preds[y_true == 1], bins=50, kde=True, color='tomato',
                 label='Churn', ax=axes[1, 1], alpha=0.6)
    axes[1, 1].set_title('OOF Predicted Probabilities by Class', fontsize=14, fontweight='bold', color='midnightblue')
    axes[1, 1].set_xlabel('Predicted Probability')
    axes[1, 1].legend()


    sns.histplot(test_preds, bins=50, kde=True, color='seagreen', ax=axes[1, 2])
    axes[1, 2].set_title('Test Set Predicted Probabilities', fontsize=14, fontweight='bold', color='midnightblue')
    axes[1, 2].set_xlabel('Predicted Probability')

    plt.tight_layout()
    plt.show()

plot_dashboard(train['Churn'], oof_final, pred_final, importances_xgb, importances_lgb, FEATURES)




# Submission

In [ ]:
submission = pd.DataFrame({'id': test['id'], 'Churn': pred_final})
submission.to_csv('submission.csv', index=False)
print("\nSubmission saved!")

Acknowledgement : 
- [https://www.kaggle.com/code/jocelyndumlao/churn-prediction-0-917-auc-pipeline](https://www.kaggle.com/code/jocelyndumlao/churn-prediction-0-917-auc-pipeline)
- [https://www.kaggle.com/datasets/cdeotte/s6e3-original-dataset](https://www.kaggle.com/datasets/cdeotte/s6e3-original-dataset)

